In [0]:
import sys
import uuid
from datetime import datetime, timezone
 
sys.path.insert(0, "/Workspace/Users/jean.zelada06@gmail.com/.bundle/bundle_dev_finpay/dev/files/src")
 
from Silver_Functions import (
    clean_transactions,
    deduplicate_by_latest,
    write_silver_event_log,
)
 
from pyspark.sql import functions as F



In [0]:
SOURCE_NAME     = "transactions"
BRONZE_TABLE    = "fintech_finpay.bronze.transactions"
SILVER_TABLE    = "fintech_finpay.silver.transactions"
QUARANTINE_TABLE= "fintech_finpay.silver.quarantine"
PK              = ["transaction_id"]
PIPELINE_RUN_ID = str(uuid.uuid4())
 
try:
    NOTEBOOK_PATH = dbutils.notebook.entry_point.getDbutils() \
        .notebook().getContext().notebookPath().get()
except Exception:
    NOTEBOOK_PATH = "silver_transactions"
 
print(f"  SOURCE         : {BRONZE_TABLE}")
print(f"  TARGET         : {SILVER_TABLE}")
print(f"  QUARANTINE     : {QUARANTINE_TABLE}")
print(f"  PIPELINE RUN   : {PIPELINE_RUN_ID}")


  SOURCE         : fintech_finpay.bronze.transactions
  TARGET         : fintech_finpay.silver.transactions
  QUARANTINE     : fintech_finpay.silver.quarantine
  PIPELINE RUN   : 9adb568b-ba25-4014-8ff2-45d17d5d0e6b


In [0]:
df1 = spark.read.format("delta").table(BRONZE_TABLE)
 
print(f"df1 — bronze leído como batch")
print(f"registros  : {df1.count():,}")
print(f"isStreaming: {df1.isStreaming}")   # debe imprimir False



df1 — bronze leído como batch
registros  : 51,150
isStreaming: False


In [0]:
# --- Paso A: deduplicar y cortar linaje ---
df_dedup = deduplicate_by_latest(df1, PK)
 
# localCheckpoint() materializa el DataFrame en el executor y rompe el linaje.
# A partir de aquí df_dedup_checkpoint es un DataFrame batch nuevo y limpio.
df_dedup_checkpoint = df_dedup.localCheckpoint()
 
print(f"Paso A — deduplicado: {df_dedup_checkpoint.count():,} registros únicos por PK")
 
# --- Paso B: limpiar y normalizar ---
df_clean = clean_transactions(df_dedup_checkpoint)
 
# --- Paso C: agregar _filter con las reglas de calidad ---
# _filter = True  → RECHAZADO (falla al menos una regla)
# _filter = False → VÁLIDO    (pasa todas las reglas)
 
df2 = df_clean.withColumn(
    "_filter",
    (
        F.col("transaction_id").isNull()
        | ~F.col("transaction_id").rlike(r"^TXN-[0-9]{8}-[0-9]{5}$")
        | F.col("user_id").isNull()
        | ~F.col("user_id").rlike(r"^USR-[0-9]{6}$")
        | F.col("merchant_id").isNull()
        | ~F.col("merchant_id").rlike(r"^MCH-[0-9]{5}$")
        | ~F.col("channel").isin("web", "app", "pos", "kiosko")
        | ~F.col("transaction_type").isin("pago", "reversa", "retiro", "payment")
        | F.col("amount").isNull()
        | (F.col("amount") <= 0)
        | F.col("transaction_date").isNull()
        | ~F.col("status").isin("aprobado", "rechazado", "pendiente", "procesado")
        | ((F.col("transaction_type") == "reversa") & F.col("reference_id").isNull())
    )
)
 
total_valid    = df2.filter(~F.col("_filter")).count()
total_rejected = df2.filter( F.col("_filter")).count()
 
print(f"df2 — limpio + columna _filter")
print(f"válidos    (_filter=False): {total_valid:,}")
print(f"rechazados (_filter=True) : {total_rejected:,}")


Paso A — deduplicado: 50,000 registros únicos por PK
df2 — limpio + columna _filter
válidos    (_filter=False): 44,978
rechazados (_filter=True) : 5,022


In [0]:
df_validados = (
    df2
    .filter(~F.col("_filter"))
    .drop("_filter")
    .withColumn("_silver_ts", F.current_timestamp())
)

print(f"df_validados  : {df_validados.count():,} registros")


df_validados  : 44,978 registros


In [0]:
# Rechazados: agregar rejection_reason para quarantine
df_rechazados = (
    df2
    .filter(F.col("_filter"))
    .withColumn(
        "rejection_reason",
        F.when(F.col("transaction_id").isNull(),
            F.lit("transaction_id: nulo crítico")
        ).when(~F.col("transaction_id").rlike(r"^TXN-[0-9]{8}-[0-9]{5}$"),
            F.lit("transaction_id: formato inválido (esperado TXN-YYYYMMDD-NNNNN)")
        ).when(F.col("user_id").isNull(),
            F.lit("user_id: nulo crítico")
        ).when(~F.col("user_id").rlike(r"^USR-[0-9]{6}$"),
            F.lit("user_id: formato inválido (esperado USR-NNNNNN)")
        ).when(F.col("merchant_id").isNull(),
            F.lit("merchant_id: nulo crítico")
        ).when(~F.col("merchant_id").rlike(r"^MCH-[0-9]{5}$"),
            F.lit("merchant_id: formato inválido (esperado MCH-NNNNN)")
        ).when(~F.col("channel").isin("web", "app", "pos", "kiosko"),
            F.concat(F.lit("channel: valor inválido '"), F.col("channel"), F.lit("' (esperado: web|app|pos|kiosko)"))
        ).when(~F.col("transaction_type").isin("pago", "reversa", "retiro", "payment"),
            F.concat(F.lit("transaction_type: valor inválido '"), F.col("transaction_type"), F.lit("' (esperado: pago|reversa|retiro|payment)"))
        ).when(F.col("amount").isNull(),
            F.lit("amount: nulo crítico")
        ).when(F.col("amount") <= 0,
            F.concat(F.lit("amount: valor no positivo ("), F.col("amount").cast("string"), F.lit(")"))
        ).when(F.col("transaction_date").isNull(),
            F.lit("transaction_date: nulo o formato no reconocido (esperado yyyy-MM-dd o dd/MM/yyyy)")
        ).when(~F.col("status").isin("aprobado", "rechazado", "pendiente", "procesado"),
            F.concat(F.lit("status: valor inválido '"), F.col("status"), F.lit("' (esperado: aprobado|rechazado|pendiente|procesado)"))
        ).when(
            (F.col("transaction_type") == "reversa") & F.col("reference_id").isNull(),
            F.lit("reference_id: nulo crítico cuando transaction_type=reversa")
        ).otherwise(F.lit("múltiples campos inválidos"))
    )
)
 
print(f"df_rechazados : {df_rechazados.count():,} registros")


df_rechazados : 5,022 registros


In [0]:
df_rechazados.createOrReplaceTempView("vw_rechazados")

display(spark.sql("""
SELECT
rejection_reason, count(*)
FROM vw_rechazados
group by rejection_reason
"""))

rejection_reason,count(*)
amount: nulo crítico,1477
status: valor inválido 'null' (esperado: aprobado|rechazado|pendiente|procesado),1010
reference_id: nulo crítico cuando transaction_type=reversa,2535


In [0]:
start_ts = datetime.now(timezone.utc)
 
# Registrar df_validados como vista temporal para usar en SQL MERGE
df_validados.createOrReplaceTempView("silver_transactions_incoming")
 
spark.sql(f"""
    MERGE INTO {SILVER_TABLE} AS target
    USING silver_transactions_incoming AS source
    ON target.transaction_id = source.transaction_id
    WHEN MATCHED THEN UPDATE SET
        target.user_id          = source.user_id,
        target.merchant_id      = source.merchant_id,
        target.channel          = source.channel,
        target.transaction_type = source.transaction_type,
        target.amount           = source.amount,
        target.currency         = source.currency,
        target.transaction_date = source.transaction_date,
        target.status           = source.status,
        target.reference_id     = source.reference_id,
        target._source_name     = source._source_name,
        target._source_format   = source._source_format,
        target._schema_version  = source._schema_version,
        target._ingestion_ts    = source._ingestion_ts,
        target._source_file     = source._source_file,
        target._silver_ts       = source._silver_ts
    WHEN NOT MATCHED THEN INSERT *
""")
 
records_processed = spark.table(SILVER_TABLE).count()
print(f"✅ MERGE completado → {SILVER_TABLE}")
print(f"   Total registros en tabla: {records_processed:,}")


✅ MERGE completado → fintech_finpay.silver.transactions
   Total registros en tabla: 44,978


In [0]:
business_cols = [
    "transaction_id", "user_id", "merchant_id", "channel",
    "transaction_type", "amount", "currency", "transaction_date",
    "status", "reference_id"
]
 
df_quarantine = (
    df_rechazados
    .withColumn("quarantine_id",   F.expr("uuid()"))
    .withColumn("event_ts",        F.current_timestamp())
    .withColumn("event_date",      F.current_date())
    .withColumn("source_name",     F.lit(SOURCE_NAME))
    .withColumn("target_table",    F.lit(SILVER_TABLE))
    .withColumn("rejected_field",  F.lit(None).cast("string"))
    .withColumn("raw_record",      F.to_json(F.struct(*business_cols)))
    .withColumn("pipeline_run_id", F.lit(PIPELINE_RUN_ID))
    .select(
        "quarantine_id", "event_ts", "event_date",
        "source_name", "target_table", "rejection_reason",
        "rejected_field", "raw_record", "pipeline_run_id",
    )
)
 
(
    df_quarantine.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)
 
records_quarantined = (
    spark.table(QUARANTINE_TABLE)
    .filter(F.col("pipeline_run_id") == PIPELINE_RUN_ID)
    .count()
)
 
print(f"✅ Quarantine → {records_quarantined:,} registros rechazados en este run")


✅ Quarantine → 5,022 registros rechazados en este run


In [0]:
duration_sec = round((datetime.now(timezone.utc) - start_ts).total_seconds(), 2)
 
print()
print("=" * 65)
print("  📊  REPORTE SILVER — transactions")
print("=" * 65)
print(f"  Pipeline run ID     : {PIPELINE_RUN_ID}")
print(f"  Bronze leídos       : {df1.count():,}")
print(f"  Tras deduplicación  : {df_dedup_checkpoint.count():,}")
print(f"  ✅ Válidos (MERGE)   : {records_processed:,}")
print(f"  ⚠️  Quarantine       : {records_quarantined:,}")
print(f"  ⏱️  Duración         : {duration_sec}s")
print("=" * 65)
 
if records_quarantined > 0:
    print("\n🔍 Muestra de rechazados en este run:")
    display(
        spark.table(QUARANTINE_TABLE)
        .filter(F.col("pipeline_run_id") == PIPELINE_RUN_ID)
        .select("source_name", "rejection_reason", "raw_record", "event_ts")
        .limit(20)
    )
 



  📊  REPORTE SILVER — transactions
  Pipeline run ID     : 9adb568b-ba25-4014-8ff2-45d17d5d0e6b
  Bronze leídos       : 51,150
  Tras deduplicación  : 50,000
  ✅ Válidos (MERGE)   : 44,978
  ⚠️  Quarantine       : 5,022
  ⏱️  Duración         : 17.23s

🔍 Muestra de rechazados en este run:


source_name,rejection_reason,raw_record,event_ts
transactions,amount: nulo crítico,"{""transaction_id"":""TXN-20240101-00012"",""user_id"":""USR-009639"",""merchant_id"":""MCH-00284"",""channel"":""web"",""transaction_type"":""payment"",""currency"":""PEN"",""transaction_date"":""2024-01-01"",""status"":""aprobado""}",2026-05-28T21:59:28.037Z
transactions,status: valor inválido 'null' (esperado: aprobado|rechazado|pendiente|procesado),"{""transaction_id"":""TXN-20240101-00017"",""user_id"":""USR-005139"",""merchant_id"":""MCH-00030"",""channel"":""app"",""transaction_type"":""pago"",""amount"":12791.53,""currency"":""PEN"",""transaction_date"":""2024-01-01"",""status"":""null""}",2026-05-28T21:59:28.037Z
transactions,amount: nulo crítico,"{""transaction_id"":""TXN-20240101-00019"",""user_id"":""USR-007492"",""merchant_id"":""MCH-00162"",""channel"":""web"",""transaction_type"":""pago"",""currency"":""CLP"",""transaction_date"":""2024-01-01"",""status"":""aprobado""}",2026-05-28T21:59:28.037Z
transactions,reference_id: nulo crítico cuando transaction_type=reversa,"{""transaction_id"":""TXN-20240101-00020"",""user_id"":""USR-006055"",""merchant_id"":""MCH-00146"",""channel"":""pos"",""transaction_type"":""reversa"",""amount"":9177.11,""currency"":""PEN"",""transaction_date"":""2024-01-01"",""status"":""rechazado""}",2026-05-28T21:59:28.037Z
transactions,reference_id: nulo crítico cuando transaction_type=reversa,"{""transaction_id"":""TXN-20240101-00057"",""user_id"":""USR-003994"",""merchant_id"":""MCH-00234"",""channel"":""app"",""transaction_type"":""reversa"",""amount"":8992.67,""currency"":""PEN"",""transaction_date"":""2024-01-01"",""status"":""aprobado""}",2026-05-28T21:59:28.037Z
transactions,reference_id: nulo crítico cuando transaction_type=reversa,"{""transaction_id"":""TXN-20240101-00059"",""user_id"":""USR-003913"",""merchant_id"":""MCH-00293"",""channel"":""app"",""transaction_type"":""reversa"",""amount"":11171.85,""currency"":""MXN"",""transaction_date"":""2024-01-01"",""status"":""aprobado""}",2026-05-28T21:59:28.037Z
transactions,reference_id: nulo crítico cuando transaction_type=reversa,"{""transaction_id"":""TXN-20240101-00064"",""user_id"":""USR-008956"",""merchant_id"":""MCH-00081"",""channel"":""app"",""transaction_type"":""reversa"",""amount"":4490.82,""currency"":""MXN"",""transaction_date"":""2024-01-01"",""status"":""aprobado""}",2026-05-28T21:59:28.037Z
transactions,amount: nulo crítico,"{""transaction_id"":""TXN-20240101-00065"",""user_id"":""USR-009418"",""merchant_id"":""MCH-00152"",""channel"":""app"",""transaction_type"":""pago"",""currency"":""MXN"",""transaction_date"":""2024-01-01"",""status"":""rechazado""}",2026-05-28T21:59:28.037Z
transactions,amount: nulo crítico,"{""transaction_id"":""TXN-20240101-00069"",""user_id"":""USR-009132"",""merchant_id"":""MCH-00188"",""channel"":""app"",""transaction_type"":""retiro"",""currency"":""MXN"",""transaction_date"":""2024-01-01"",""status"":""aprobado""}",2026-05-28T21:59:28.037Z
transactions,reference_id: nulo crítico cuando transaction_type=reversa,"{""transaction_id"":""TXN-20240101-00090"",""user_id"":""USR-007369"",""merchant_id"":""MCH-00466"",""channel"":""pos"",""transaction_type"":""reversa"",""amount"":9077.06,""currency"":""MXN"",""transaction_date"":""2024-01-01"",""status"":""pendiente""}",2026-05-28T21:59:28.037Z


In [0]:
status_final = "SUCCESS" if records_processed > 0 or records_quarantined >= 0 else "FAILED"
 
write_silver_event_log(
    spark               = spark,
    source_name         = SOURCE_NAME,
    target_table        = SILVER_TABLE,
    status              = status_final,
    records_processed   = records_processed,
    records_quarantined = records_quarantined,
    duration_sec        = duration_sec,
    pipeline_run_id     = PIPELINE_RUN_ID,
    notebook_path       = NOTEBOOK_PATH,
)
 
print(f"✅ Event log → fintech_finpay.observability.pipeline_event_log")


2026-05-28 21:59:36,868 [INFO]   [OBS] Event log silver persistido → fintech_finpay.observability.pipeline_event_log | status=SUCCESS | processed=44,978 | quarantined=5,022


✅ Event log → fintech_finpay.observability.pipeline_event_log


In [0]:
display(
    spark.sql(
        """
        SELECT COUNT(*) as Q_registros,
        "bronze_transactions" AS TABLA
        FROM fintech_finpay.bronze.transactions
        UNION
        SELECT COUNT(*) as Q_registros,
        "silver_transactions" AS TABLA
        FROM fintech_finpay.silver.transactions
        UNION
        SELECT COUNT(*) as Q_registros,
        "silver_quarantine" AS TABLA
        FROM fintech_finpay.silver.quarantine
        """
    )
)

Q_registros,TABLA
51150,bronze_transactions
44978,silver_transactions
5022,silver_quarantine
